# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
meta = dataset.metadata

print(f"Dataset name: {meta.name}\nDescription: {meta.description}\nVersion: {meta.version}\nIdentifier: {meta.identifier}")

## 2. Data Overview
Review available record sets, their fields/columns, and their `@id`s.

In [ ]:
# List all record sets and their fields using Croissant metadata
print('Available record sets:')
if not hasattr(meta, 'record_sets') or not meta.record_sets:
    print('No record sets found in metadata. Attempting to enumerate automatically...')
    # Fallback for auto-discovery (some schemas place record sets in .records)
    record_sets = [rs for rs in dataset.list_record_sets()]
else:
    record_sets = meta.record_sets

if not isinstance(record_sets, list):
    record_sets = [record_sets]

for rs in record_sets:
    rs_id = rs["@id"] if isinstance(rs, dict) and "@id" in rs else rs
    print(f'  - Record set @id: {rs_id}')
    try:
        fields = dataset.record_set_schema(rs_id)["fields"]
        for f in fields:
            f_id = f["@id"] if isinstance(f, dict) and "@id" in f else f
            name = f.get('name', '') if isinstance(f, dict) else ''
            print(f'      - Field @id: {f_id}  (name: {name})')
    except Exception as e:
        print('    -> Could not get fields:', e)
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all available record set ids
record_set_ids = dataset.list_record_sets()
print(f"Record set @ids: {record_set_ids}")

dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nExtracting records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Fields: {list(df.columns)}")
        print(df.head(2))
    except Exception as e:
        print(f"  Cannot extract records from {record_set_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping data. Example below operates on key numeric fields from the main record set.

In [ ]:
# Pick the main protein abundance record set by @id
# You can change `main_record_set_id` to a different id as needed
main_record_set_id = None
for rsid in dataframes:
    # Heuristic: pick record sets containing "protein", "abundance", or just the first record set
    if 'protein' in rsid.lower() or 'abundance' in rsid.lower():
        main_record_set_id = rsid
        break
if main_record_set_id is None and dataframes:
    main_record_set_id = list(dataframes)[0]

df = dataframes[main_record_set_id].copy() if main_record_set_id is not None else pd.DataFrame()

print(f"\nUsing record set for EDA: {main_record_set_id}")

numeric_candidates = df.select_dtypes('number').columns.tolist()
if not numeric_candidates:
    # Try to coax columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            continue
    numeric_candidates = df.select_dtypes('number').columns.tolist()

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    print("No numeric columns found - can't proceed with numeric EDA.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold} (showing up to 5 rows):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    # Try grouping by a categorical field, e.g., accession or description columns
    group_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the main numeric field and relationship to a key categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field} in Record Set: {main_record_set_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot grouped by group_field if available
    if 'group_field' in locals() and group_field is not None:
        n_categories = df[group_field].nunique()
        if n_categories > 1 and n_categories < 30:
            plt.figure(figsize=(12, 5))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated the use of `mlcroissant` to explore the FAIR² dataset on mass spectrometry analysis of extracellular vesicles. We loaded metadata, surveyed record sets and fields, extracted data with field and record set `@id` references, performed basic normalization, filtering, and produced visualizations of key numeric fields for further biological/statistical analysis.

**Next steps:** You can extend this notebook by exploring more record sets or applying more advanced machine learning and statistical techniques using the structured data loaded above.